# 🧬 Submission 3: Hybrid Dual-Objective Pipeline
**Strategy:** Task-Specific Feature Selection + Unsupervised ESM Bottlenecks

This iteration combines the strengths of our previous pipelines. Instead of forcing a single set of features to solve both classification and ranking, we introduce **Dual-Objective Feature Selection**—allowing the classifier and regressor to independently mine the data for their distinct goals. Furthermore, we safely reintroduce ESM-2 embeddings via an unsupervised PCA bottleneck to capture sequence biophysics without leaking evolutionary lineage.

In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/retroviral-challenge-predict/family_splits.csv
/kaggle/input/competitions/retroviral-challenge-predict/sample_submission.csv
/kaggle/input/competitions/retroviral-challenge-predict/esm2_embeddings.npz
/kaggle/input/competitions/retroviral-challenge-predict/feature_dictionary.csv
/kaggle/input/competitions/retroviral-challenge-predict/train.csv
/kaggle/input/competitions/retroviral-challenge-predict/test.csv
/kaggle/input/competitions/retroviral-challenge-predict/structures/AVIRE-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Gs-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Drg-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Ec48-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Gate-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/Ne144-RT.pdb
/kaggle/input/competitions/retroviral-challenge-predict/structures/RSVSB-RT.pdb
/kaggle/i

In [2]:
import os
import numpy as np
import pandas as pd
import warnings
from scipy.stats import rankdata
from scipy.special import expit as sigmoid
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR
from sklearn.ensemble import VotingRegressor, VotingClassifier, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.metrics import average_precision_score
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

warnings.filterwarnings('ignore')

## 📐 1. Evaluation Metrics (CLS)
We implement the competition's Custom Cross-Lineage Score (CLS). To ensure our continuous models properly prioritize top-tier enzymes, we utilize custom `sample_weight` arrays defined as `y_true + 0.01` during the `.fit()` stage to mathematically align with the Weighted Spearman penalty.

In [3]:
# ==========================================
# 1. EVALUATION METRICS (CLS)
# ==========================================

def calculate_weighted_spearman(y_true_eff: np.ndarray, y_pred: np.ndarray) -> float:
    weights = y_true_eff + 0.01
    rank_true = rankdata(y_true_eff)
    rank_pred = rankdata(y_pred)
    mean_true = np.average(rank_true, weights=weights)
    mean_pred = np.average(rank_pred, weights=weights)
    num = np.sum(weights * (rank_true - mean_true) * (rank_pred - mean_pred))
    den_true = np.sum(weights * (rank_true - mean_true)**2)
    den_pred = np.sum(weights * (rank_pred - mean_pred)**2)
    if den_true == 0 or den_pred == 0: return 0.0
    corr = num / np.sqrt(den_true * den_pred)
    return max(0.0, float(corr))

def calculate_cls(y_true_active, y_true_eff, y_pred):
    pr_auc = average_precision_score(y_true_active, y_pred)
    w_spearman = calculate_weighted_spearman(y_true_eff, y_pred)
    if pr_auc <= 0 or w_spearman <= 0: return 0.0, pr_auc, w_spearman
    cls_score = 2 * (pr_auc * w_spearman) / (pr_auc + w_spearman)
    return cls_score, pr_auc, w_spearman

## 🛠️ 2. Safe Feature Engineering & Leak Guard
We synthesize highly localized features, safely mapping structural log-likelihoods to a pseudo-probability scale via `np.exp()`. 

**Strict Leakage Guard:** All sequence data, family strings, and `foldseek_TM_*` lineage markers are aggressively stripped from the dataset to mathematically prevent evolutionary memorization.

In [4]:
# ==========================================
# 2. FEATURE ENGINEERING 
# ==========================================

def engineer_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    if "triad_detected" in train_df.columns and "esmif_log_likelihood" in train_df.columns:
        for df in [train_df, test_df]:
            ll = df["esmif_log_likelihood"].fillna(df["esmif_log_likelihood"].median())
            # Use np.exp to map to a pseudo-probability scale. StandardScaler handles the rest safely!
            df["feat_catalytic_fitness"] = df["triad_detected"].fillna(0) * np.exp(ll)

    if "triad_rmsd_to_hiv" in train_df.columns:
        for df in [train_df, test_df]:
            df["feat_active_site_quality"] = 1.0 / (1.0 + df["triad_rmsd_to_hiv"].fillna(99.0))

    return train_df, test_df

def load_competition_data(base_path):
    train_df = pd.read_csv(os.path.join(base_path, "train.csv"))
    test_df = pd.read_csv(os.path.join(base_path, "test.csv"))
    train_df.columns = train_df.columns.str.strip()
    test_df.columns = test_df.columns.str.strip()
    
    eff_col = next((c for c in train_df.columns if 'effic' in c.lower() or 'pct' in c.lower()), None)
    active_col = next((c for c in train_df.columns if 'active' in c.lower()), None)

    train_df, test_df = engineer_features(train_df, test_df)
    
    esm_features = []
    esm_path = os.path.join(base_path, "esm2_embeddings.npz")
    if os.path.exists(esm_path):
        esm2_data = np.load(esm_path, allow_pickle=True)
        esm_df = pd.DataFrame(esm2_data['embeddings'], columns=[f'esm_{i}' for i in range(1280)])
        esm_df['rt_name'] = esm2_data['names']
        train_df = train_df.merge(esm_df, on='rt_name', how='left')
        test_df = test_df.merge(esm_df, on='rt_name', how='left')
        esm_features = [f'esm_{i}' for i in range(1280)]
    
    leaky_prefixes = ['foldseek_TM_', 'rt_family', 'rt_name', 'sequence']
    handcrafted_features = [
        c for c in train_df.columns 
        if c not in esm_features and c not in [eff_col, active_col]
        and not any(str(c).startswith(leak) for leak in leaky_prefixes)
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]
    return train_df, test_df, handcrafted_features, esm_features, eff_col, active_col


## ⚖️ 3. Metric-Aligned Custom Estimators
Standard machine learning regressors optimize for uniform Mean Squared Error (MSE). We wrap `Ridge`, `SVR`, and `GaussianProcessRegressor` into metric-aligned custom estimators. By targeting the log-transformed efficiency (`np.log1p`) and fitting against our Custom Rank weights, these models natively approximate the Kaggle leaderboard metric.

In [5]:
# ==========================================
# 3. METRIC ALIGNED MODELS (The Champions)
# ==========================================

class LogMetricAlignedRidge(RegressorMixin, BaseEstimator):
    def __init__(self, alpha=15.0): self.alpha = alpha
    def fit(self, X, y, **fit_params):
        self.model = Ridge(alpha=self.alpha, random_state=42)
        weights = np.asarray(y) + 0.01
        self.model.fit(X, np.log1p(y), sample_weight=weights)
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))

class LogMetricAlignedSVR(RegressorMixin, BaseEstimator):
    def __init__(self, C=2.0, epsilon=0.05): self.C = C; self.epsilon = epsilon
    def fit(self, X, y, **fit_params):
        self.model = SVR(kernel='rbf', C=self.C, epsilon=self.epsilon)
        weights = np.asarray(y) + 0.01
        self.model.fit(X, np.log1p(y), sample_weight=weights)
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))

class LogMetricAlignedGP(RegressorMixin, BaseEstimator):
    def __init__(self): pass
    def fit(self, X, y, **fit_params):
        weights = np.asarray(y) + 0.01
        noise_variance = 0.1 / weights 
        kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=1.5)
        self.model = GaussianProcessRegressor(kernel=kernel, alpha=noise_variance, n_restarts_optimizer=5, random_state=42)
        self.model.fit(X, np.log1p(y))
        return self
    def predict(self, X): return np.maximum(0.0, np.expm1(self.model.predict(X)))


## 🧠 4. Decoupled Dual-Objective Architecture
This is the core innovation of this pipeline:
* **Distinct Feature Selection:** The Classification branch uses `f_classif` ($K=10$) to hunt for sharp binary structural markers, while the Regression branch uses `f_regression` ($K=15$) to find continuous efficiency slopes.
* **Missingness as Signal:** We retain `add_indicator=True` to allow models to learn from ESMFold's structural resolution failures.
* **ESM-2 PCA Bottleneck:** 1280-d language embeddings are squashed to 6 principal components, extracting global 3D biophysics while destroying phylogenetic leakage.

In [6]:
# ==========================================
# 4. DECOUPLED DUAL-OBJECTIVE ARCHITECTURE
# ==========================================

def build_feature_pipeline(handcrafted_features, esm_features, task='regression'):
    """Builds a preprocessor explicitly mathematically tuned for Classification OR Regression"""
    transformers = []
    
    # 🌟 THE UPGRADE: Distinct 3D/Tabular feature selection for distinct tasks!
    score_function = f_classif if task == 'classification' else f_regression
    k_feats = 10 if task == 'classification' else 15 # Classifier needs fewer, sharper 3D signals
    
    hc_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=score_function, k=k_feats)) 
    ])
    transformers.append(('handcrafted', hc_pipe, handcrafted_features))
    
    if esm_features:
        esm_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
            ('scaler1', StandardScaler()),
            ('pca', PCA(n_components=6, random_state=42)), # Safe, Unsupervised OOD logic restored
            ('scaler2', StandardScaler())
        ])
        transformers.append(('esm2', esm_pipe, esm_features))
        
    return ColumnTransformer(transformers)

class DecoupledExpectedValuePredictor(RegressorMixin, BaseEstimator):
    """
    Independently trains a classification pipeline and a regression pipeline on the RAW data, 
    allowing each to isolate its own scientifically optimal features!
    """
    def __init__(self, classifier_pipe, regressor_pipe):
        self.classifier_pipe = classifier_pipe
        self.regressor_pipe = regressor_pipe

    def fit(self, X, y, **fit_params):
        y_active = (np.asarray(y) > 0).astype(int)
        
        self.classifier_pipe_ = clone(self.classifier_pipe)
        self.classifier_pipe_.fit(X, y_active)
        
        self.regressor_pipe_ = clone(self.regressor_pipe)
        self.regressor_pipe_.fit(X, y, **fit_params)
        return self

    def predict(self, X):
        p_active = self.classifier_pipe_.predict_proba(X)[:, 1]
        pred_eff = self.regressor_pipe_.predict(X)
        return (p_active ** 2) * pred_eff

def build_model_pipeline(handcrafted_features, esm_features):
    # 1. Classification Branch (Hunts for PR-AUC structural markers)
    clf_prep = build_feature_pipeline(handcrafted_features, esm_features, task='classification')
    
    # 🌟 APEX GATEKEEPER: Linear logic + Non-linear shallow tree logic
    clf1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.2, class_weight='balanced', random_state=42)
    clf2 = RandomForestClassifier(n_estimators=50, max_depth=2, class_weight='balanced', random_state=42)
    gatekeeper_clf = VotingClassifier(estimators=[('lr', clf1), ('rf', clf2)], voting='soft')
    
    clf_pipe = Pipeline([('prep', clf_prep), ('clf', gatekeeper_clf)])
    
    # 2. Regression Branch (Hunts for Weighted Spearman continuous slopes)
    reg_prep = build_feature_pipeline(handcrafted_features, esm_features, task='regression')
    continuous_ranker = VotingRegressor(
        estimators=[
            ('ridge', LogMetricAlignedRidge(alpha=15.0)),
            ('svr', LogMetricAlignedSVR(C=2.0, epsilon=0.05)),
            ('gp', LogMetricAlignedGP())
        ],
        weights=[0.50, 0.20, 0.30] 
    )
    reg_pipe = Pipeline([('prep', reg_prep), ('reg', continuous_ranker)])
    
    return DecoupledExpectedValuePredictor(classifier_pipe=clf_pipe, regressor_pipe=reg_pipe)


## 🔄 5. LOFO Evaluation & Smart Inference
To accurately measure Phase 2 Wet-Lab generalization, we evaluate via strict Leave-One-Family-Out (LOFO) validation, now with verbose fold logging to track lineage sizes. 

For the final output, we utilize **Smart Inference Mapping**: Known public sequences receive their honest Out-Of-Fold predictions (securing our Phase 1 Kaggle rank), while unseen Phase 2 proprietary sequences dynamically trigger predictions from our finalized full-dataset pipeline.

In [7]:
# ==========================================
# 5. LOFO HARNESS & KAGGLE EXPORT
# ==========================================

def run_lofo_validation(df_train, handcrafted_feats, esm_feats, eff_col, active_col):
    print("\n--- Starting Leak-Free Leave-One-Family-Out (LOFO) Cross-Validation ---")
    families = df_train['rt_family'].unique()
    oof_predictions = np.zeros(len(df_train))
    
    for fold, family in enumerate(families):
        val_mask = df_train['rt_family'] == family
        train_mask = ~val_mask
        X_train, y_train = df_train[train_mask], df_train.loc[train_mask, eff_col]
        X_val = df_train[val_mask]
        
        pipeline = build_model_pipeline(handcrafted_feats, esm_feats)
        pipeline.fit(X_train, y_train)
        oof_predictions[val_mask] = pipeline.predict(X_val)
        print(f"Fold {fold+1}/7 | Held Out: {family:<25} | Val Size: {val_mask.sum()}")

    y_true_active = df_train[active_col].values
    y_true_eff = df_train[eff_col].values
    cls_score, pr_auc, w_spearman = calculate_cls(y_true_active, y_true_eff, oof_predictions)
    
    print("\n" + "="*45)
    print(" LOFO POOLED EVALUATION (NO LEAKS)")
    print("="*45)
    print(f"PR-AUC (Classification)     : {pr_auc:.4f}")
    print(f"Weighted Spearman (Ranking) : {w_spearman:.4f}")
    print(f"Final CLS (Harmonic Mean)   : {cls_score:.4f}")
    print("="*45 + "\n")
    return oof_predictions

# REPLACE your __main__ block:
if __name__ == "__main__":
    KAGGLE_PATH = "/kaggle/input/competitions/retroviral-challenge-predict"
    if not os.path.exists(KAGGLE_PATH): KAGGLE_PATH = "." 
    
    train_df, test_df, hc_feats, esm_feats, target_eff, target_act = load_competition_data(KAGGLE_PATH)
    
    # 1. Run strict LOFO to get leaderboard mappings
    oof_preds = run_lofo_validation(train_df, hc_feats, esm_feats, target_eff, target_act)
    
    # 2. Train the Final Generalization Model
    print("Training Final Phase 2 Model on 100% available data...")
    final_pipeline = build_model_pipeline(hc_feats, esm_feats)
    final_pipeline.fit(train_df, train_df[target_eff])
    
    # 3. Generate native predictions for unseen Phase 2 data!
    phase2_predictions = final_pipeline.predict(test_df)
    
    # 4. Smart Mapping Logic
    print("Mapping True OOF Predictions and Phase 2 Generalization...")
    oof_dict = dict(zip(train_df['rt_name'], oof_preds))
    final_preds = []
    
    for idx, row in test_df.iterrows():
        name = row['rt_name']
        if name in oof_dict:
            final_preds.append(oof_dict[name])      # Phase 1 Kaggle CV accuracy
        else:
            final_preds.append(phase2_predictions[idx]) # Phase 2 Wet-Lab prediction
            
    submission = pd.DataFrame({'rt_name': test_df['rt_name'], 'predicted_score': final_preds})
    submission.to_csv("submission.csv", index=False)
    print("✅ Created 'submission.csv'.")


--- Starting Leak-Free Leave-One-Family-Out (LOFO) Cross-Validation ---
Fold 1/7 | Held Out: CRISPR-associated         | Val Size: 5
Fold 2/7 | Held Out: Group_II_Intron           | Val Size: 5
Fold 3/7 | Held Out: LTR_Retrotransposon       | Val Size: 11
Fold 4/7 | Held Out: Other                     | Val Size: 5
Fold 5/7 | Held Out: Retron                    | Val Size: 12
Fold 6/7 | Held Out: Retroviral                | Val Size: 18
Fold 7/7 | Held Out: Unclassified              | Val Size: 1

 LOFO POOLED EVALUATION (NO LEAKS)
PR-AUC (Classification)     : 0.6309
Weighted Spearman (Ranking) : 0.3975
Final CLS (Harmonic Mean)   : 0.4877

Training Final Phase 2 Model on 100% available data...
Mapping True OOF Predictions and Phase 2 Generalization...
✅ Created 'submission.csv'.
